In [16]:
from langchain_groq import ChatGroq
import os
from load_dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("API KEY FETCHED")
else:
    raise ValueError("GROQ_API_KEY Not Found!!")

llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

llm_groq.invoke("Hello!")

API KEY FETCHED


AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 37, 'total_tokens': 60, 'completion_time': 0.024591436, 'completion_tokens_details': None, 'prompt_time': 0.002710998, 'prompt_tokens_details': None, 'queue_time': 0.158849976, 'total_time': 0.027302434}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d5742-41e3-76a0-9c2f-4d1cca3d3044-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 23, 'total_tokens': 60})

In [17]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag : Literal["Positive" , "Negative"]

llm_structured_output = llm_groq.with_structured_output(llm_schema)

# **CHAIN WITH CONDITIONAL CHAINS**

In [18]:
# Task 1 - Prompt

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative : {input}")
])

In [19]:
# Task 2 - LLM

llm_structured_output = llm_groq.with_structured_output(llm_schema)

In [21]:
# Task 3 - Custom Runnable

from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)->str:

    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

# **CONDITIONAL CHAIN 1**

In [22]:
# Task 1 - Prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Linkedin: {text}")
])

# Task 2 - LLM

llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

# Task 3 - String Parser

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_groq | str_parser

# **CONDITIONAL CHAIN 2**

In [23]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [24]:
def insta_chain(text:dict):

    text : text["text"]

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Instagram post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # Task 2 - LLM

    llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

    # Task 3 - String Parser

    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_groq | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

# **Final Orchestration**

In [25]:
from langchain_core.load.dump import default
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [31]:
final_orchestrator.invoke({"input":"I loved the KGF Movie, it's just Wow!!"})

'Here\'s a possible Instagram post:\n\n**Post:** \n"Good vibes only! Starting the day with a positive attitude and a heart full of gratitude. Let\'s spread kindness and love wherever we go! #PositiveVibes #Gratitude #GoodMood"\n\n**Image suggestion:** \nA bright and colorful photo with a beautiful landscape, flowers, or a happy quote to match the positive theme.\n\n**Additional ideas:**\n\n- Share a motivational quote related to positivity.\n- Use a fun and colorful font to highlight the positive message.\n- Encourage your followers to share their own positive affirmations in the comments.\n- Use relevant hashtags to reach a wider audience, such as #Motivation, #Inspiration, #Wellness, etc.'